In [ ]:
# 1. Install necessary libraries (takes ~2 mins)
!pip install git+https://github.com/huggingface/transformers.git@096f25ae1f501a084d8ff2dcaf25fbc2bd60eba4
!pip install -q -U torchao
!pip install -q trl accelerate peft

In [ ]:
# 2. Imports
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from google.colab import drive, runtime

In [ ]:
# 3. Mount Google Drive and create backup folder
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/Colab Notebooks/BitNet_CMDB_Model'
os.makedirs(drive_path, exist_ok=True)
print(f'Backup path ready: {drive_path}')

In [ ]:
# 4. Configuration
model_id = 'microsoft/bitnet-b1.58-2B-4T-bf16'  # Or the specific 1.58b variant
dataset_file = '/content/drive/MyDrive/Colab Notebooks/complex_cmdb_synthetic.jsonl'
adapter_output_dir = './bitnet-cmdb-final-adapter'
merged_output_dir = './bitnet-cmdb-final-merged'

In [ ]:
# 5. Load dataset, model, and tokenizer
print('Loading CMDB Dataset...')
dataset = load_dataset('json', data_files=dataset_file, split='train')

print('Loading BitNet Model (this may take a few minutes)...')
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Using bfloat16 for training stability on L4/A100 GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Load your model as before (ensure it's the -bf16 version)
# model = AutoModelForCausalLM.from_pretrained(...)

# 2. Crucial Step: Prepare the model for training
model = prepare_model_for_kbit_training(model)

# 3. Define the LoRA Config
# We target the 'down_proj', 'up_proj', and 'gate_proj' which are standard in BitNet's FFN
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["down_proj", "up_proj", "gate_proj", "q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Wrap the model
model = get_peft_model(model, peft_config)

# Verify that we have trainable parameters
model.print_trainable_parameters()

In [ ]:
# 6. Train and save model
def formatting_prompts_func(example):
    # This creates a single string for each row in your dataset
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}"
    )
    return text # Return the string, NOT a list

sft_config = SFTConfig(
    output_dir=adapter_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    formatting_func=formatting_prompts_func,
)

print('Starting Training Run...')
trainer.train()
trainer.save_model(adapter_output_dir)
tokenizer.save_pretrained(adapter_output_dir)

In [ ]:
# 2. Merge LoRA weights into the base model
print("Merging weights... this takes a moment.")
merged_model = model.merge_and_unload()
# 3. Save the final integrated model + tokenizer
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

In [ ]:
# 7. Archive and copy model to Google Drive
archive_name = "bitnet_cmdb_model.tar.gz"
print('Compressing and moving model to Google Drive...')
!tar -czvf {archive_name} {merged_output_dir}
!cp "{archive_name}" "{drive_path}"
print(f'Model successfully backed up to: {drive_path}')

## Optional: Push to Hugging Face Hub

This step is optional — it publishes the merged model to a private Hugging Face repo as an off-machine backup. It is **not required** for the local GGUF conversion + `bitnet.cpp` serving pipeline (see README steps 5-6).

In [ ]:
from huggingface_hub import notebook_login

# 9. Log in to Hugging Face and push model
print('Logging in to Hugging Face...')
notebook_login()

# Define your Hugging Face repository ID
# Replace 'your-username' with your actual Hugging Face username
# and 'bitnet-cmdb-final' with your desired repository name
hf_repo_id = "nirving/bitnet-cmdb-final"

print(f'Pushing model and tokenizer to Hugging Face repository: {hf_repo_id}')
merged_model.push_to_hub(hf_repo_id)
tokenizer.push_to_hub(hf_repo_id)

print('Model and tokenizer successfully pushed to Hugging Face.')

In [ ]:
# 8. Optional: auto-disconnect runtime to save Colab units
print('Shutting down runtime to save compute units...')
runtime.unassign()